## Task 1: Pre-processing

### Initial CSV Manipulation

In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import box

gdf = gpd.read_file("/kaggle/input/non-processed/india_points_preprocessed (1).gpkg")

# project to meters (important)
gdf = gdf.to_crs(3857)

# grid size (50 km)
cell = 50000

xmin, ymin, xmax, ymax = gdf.total_bounds

cols = np.arange(xmin, xmax, cell)
rows = np.arange(ymin, ymax, cell)

polygons = []
ids = []

for i, x in enumerate(cols):
    for j, y in enumerate(rows):
        polygons.append(box(x, y, x+cell, y+cell))   # <-- FIXED
        ids.append(f"{i}_{j}")

grid = gpd.GeoDataFrame({"grid_id": ids}, geometry=polygons, crs=gdf.crs)


joined = gpd.sjoin(gdf, grid, how="inner", predicate="within")


joined["is_migratory"] = joined["india_role"].str.contains("Migrant", na=False).astype(int)


obs_density = (
    joined.groupby("grid_id")
    .size()
    .reset_index(name="obs_density")
)


species_richness = (
    joined.groupby("grid_id")["taxonomic_group"]
    .nunique()
    .reset_index(name="species_richness")
)


threat_mean = (
    joined.groupby("grid_id")["iucn_score"]
    .mean()
    .reset_index(name="threat_mean")
)


migration_ratio = (
    joined.groupby("grid_id")["is_migratory"]
    .mean()
    .reset_index(name="migration_ratio")
)


from functools import reduce

dfs = [obs_density, species_richness, threat_mean, migration_ratio]

aggregated = reduce(lambda l,r: l.merge(r,on="grid_id",how="outer"), dfs)

aggregated.head()


# save aggregated table as CSV
aggregated.to_csv("aggregated_values.csv", index=False)

print("Saved aggregated_values.csv")


import matplotlib.pyplot as plt

cols = ["obs_density","species_richness","threat_mean","migration_ratio"]

aggregated[cols].hist(figsize=(10,7), bins=30)
plt.suptitle("Distribution of Spatial Variables")
plt.tight_layout()
plt.show()


import pandas as pd

# --- 1. Replace NaN in threat_mean with its mean ---
aggregated["threat_mean"] = aggregated["threat_mean"].fillna(aggregated["threat_mean"].mean())

# --- 2. Drop remaining NaNs in other columns ---
aggregated_clean = aggregated.dropna()

# --- 3. Remove threat_mean column ---
aggregated_clean = aggregated_clean.drop(columns=["threat_mean"])

# --- 4. Save new CSV ---
aggregated_clean.to_csv("aggregated_clean_no_threat.csv", index=False)

print("Saved aggregated_clean_no_threat.csv")

### For adding Latitudes and Longitudes

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box

# -------------------------
# 1) Recreate grid EXACTLY
# -------------------------
gdf = gpd.read_file("/kaggle/input/non-processed/india_points_preprocessed (1).gpkg").to_crs(3857)

cell = 50000

xmin, ymin, xmax, ymax = gdf.total_bounds
cols = np.arange(xmin, xmax, cell)
rows = np.arange(ymin, ymax, cell)

polygons = []
ids = []

for i, x in enumerate(cols):
    for j, y in enumerate(rows):
        polygons.append(box(x, y, x+cell, y+cell))
        ids.append(f"{i}_{j}")

grid = gpd.GeoDataFrame({"grid_id": ids}, geometry=polygons, crs=3857)

# -------------------------
# 2) Get centroid (lat/lon)
# -------------------------
centroids = grid.copy()
centroids["geometry"] = centroids.centroid
centroids = centroids.to_crs(4326)

centroids["latitude"] = centroids.geometry.y
centroids["longitude"] = centroids.geometry.x

centroids = centroids[["grid_id","latitude","longitude"]]

# -------------------------
# 3) Merge with CSV
# -------------------------
df = pd.read_csv("aggregated_clean_no_threat.csv")

final = df.merge(centroids, on="grid_id", how="left")

# -------------------------
# 4) Save aligned dataset
# -------------------------
final.to_csv("aggregated_with_latlon.csv", index=False)

print("DONE — perfectly aligned coordinates added")
print(final.head())

### Converting CSV to GPKG

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Load the CSV file
file_path = '/kaggle/input/pre-processed-2/aggregated_with_latlon.csv'
df = pd.read_csv(file_path)

# 2. Convert to a GeoDataFrame
# Note: longitude corresponds to X and latitude corresponds to Y
gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs="EPSG:4326"  # Standard WGS84 Coordinate Reference System
)

# 3. Export to GeoPackage format
output_file = '/kaggle/working/aggregated_bird_data.gpkg'
gdf.to_file(output_file, driver='GPKG')

print(f"File successfully converted and saved as: {output_file}")

## Task 2: Finding Spatial Autocorrelation using Global and Local Measures

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import geopandas as gpd
import libpysal
from esda.moran import Moran, Moran_Local
from esda.geary import Geary
from esda.getisord import G_Local
import numpy as np

# 1. Define your path
gpkg_path = r'C:\Users\Darsh Veer Singh\OneDrive - iiit-b\Sem 6\AID843 - Spatio Temporal Data Analytics I\Assignments\Assignment 1\biodiversity_grid_FIXED.gpkg'

print("Silently loading GeoPackage...")
gdf = gpd.read_file(gpkg_path)

print("Calculating spatial weights...")
w = libpysal.weights.Queen.from_dataframe(gdf)
w.transform = 'R'

y = gdf['species_richness'].values.astype(float)


# =========================================================
# 1. GLOBAL MEASURES
# =========================================================
print("\n--- GLOBAL STATS ---")
moran_global = Moran(y, w)
geary_global = Geary(y, w)

print(f"Global Moran's I: {moran_global.I:.4f} (p-value: {moran_global.p_sim:.4f})")
print(f"Global Geary's C: {geary_global.C:.4f} (p-value: {geary_global.p_sim:.4f})")


# =========================================================
# 2. LOCAL MEASURES
# =========================================================
print("\nCalculating Local Moran's I (LISA)...")
moran_loc = Moran_Local(y, w)
sig_level = 0.05

lisa_labels = {0: "Not Significant", 1: "High-High (Hotspot)", 2: "Low-High (Outlier)", 
               3: "Low-Low (Coldspot)", 4: "High-Low (Outlier)"}

spots = moran_loc.q.copy()
spots[moran_loc.p_sim > sig_level] = 0
gdf['LISA_Cluster'] = [lisa_labels[i] for i in spots]

print("Calculating Local Getis-Ord Gi*...")
gi_star = G_Local(y, w, transform='R', star=True)

gi_clusters = []
for p, z in zip(gi_star.p_sim, gi_star.Zs):
    if p < sig_level and z > 1.96:
        gi_clusters.append("Hotspot")
    elif p < sig_level and z < -1.96:
        gi_clusters.append("Coldspot")
    else:
        gi_clusters.append("Not Significant")
        
gdf['Gi_Star_Cluster'] = gi_clusters


# =========================================================
# 3. SAVE SILENTLY TO GEOPACKAGE
# =========================================================
output_layer = "task2_spatial_stats"
print(f"\nAppending results directly to layer: '{output_layer}'...")

# Write the geodataframe with the new categorical columns back to the GPKG
gdf.to_file(gpkg_path, layer=output_layer, driver="GPKG")
print("SUCCESS! The layer has been added.")

Silently loading GeoPackage...
Calculating spatial weights...
('WARNING: ', 28, ' is an island (no neighbors)')
('WARNING: ', 71, ' is an island (no neighbors)')
('WARNING: ', 112, ' is an island (no neighbors)')
('WARNING: ', 116, ' is an island (no neighbors)')
('WARNING: ', 117, ' is an island (no neighbors)')
('WARNING: ', 173, ' is an island (no neighbors)')
('WARNING: ', 184, ' is an island (no neighbors)')
('WARNING: ', 302, ' is an island (no neighbors)')
('WARNING: ', 391, ' is an island (no neighbors)')
('WARNING: ', 463, ' is an island (no neighbors)')
('WARNING: ', 519, ' is an island (no neighbors)')
('WARNING: ', 540, ' is an island (no neighbors)')
('WARNING: ', 541, ' is an island (no neighbors)')
('WARNING: ', 578, ' is an island (no neighbors)')
('WARNING: ', 583, ' is an island (no neighbors)')
('WARNING: ', 587, ' is an island (no neighbors)')
('WARNING: ', 590, ' is an island (no neighbors)')
('WARNING: ', 674, ' is an island (no neighbors)')
('WARNING: ', 678, ' i

## Task 4: ML Regression Models

### Spatial Negative Binomial Regression model with neighbourhood autocorrelation

In [ ]:
##### ============================================================
# Spatial Biodiversity Model (Negative Binomial + Neighbours)
# ============================================================

import geopandas as gpd
import numpy as np
import statsmodels.api as sm
from libpysal.weights import Queen
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------------------------------------------------
# 1. Load dataset (grid cells with richness & features)
# ------------------------------------------------------------
gdf = gpd.read_file("biodiversity_grid_FIXED.gpkg")

# remove invalid rows
gdf = gdf.dropna(subset=["species_richness"])
gdf = gdf[gdf["obs_density"] > 0].copy()

print("Total cells:", len(gdf))

# ------------------------------------------------------------
# 2. Build spatial neighbors
# ------------------------------------------------------------
w = Queen.from_dataframe(gdf)
w.transform = 'r'

# neighbour richness feature
gdf["neighbor_richness"] = w.sparse @ gdf["species_richness"].values

# ------------------------------------------------------------
# 3. Train Negative Binomial spatial regression
# ------------------------------------------------------------
y = gdf["species_richness"]

X = gdf[["obs_density","migration_ratio","neighbor_richness"]]
X = sm.add_constant(X, has_constant='add')

model = sm.GLM(y, X, family=sm.families.NegativeBinomial()).fit()

print(model.summary())

# predictions
gdf["predicted_richness"] = model.predict(X)

# ------------------------------------------------------------
# 4. Spatial accuracy test (20% hidden cells)
# ------------------------------------------------------------
np.random.seed(42)
mask = np.random.rand(len(gdf)) < 0.2

train = gdf[~mask].copy()
test  = gdf[mask].copy()

print("Train:", len(train), "Test:", len(test))

# rebuild neighbors only on training area
w_train = Queen.from_dataframe(train)
w_train.transform = 'r'

train["neighbor_richness"] = w_train.sparse @ train["species_richness"].values

# retrain model
y_train = train["species_richness"]
X_train = train[["obs_density","migration_ratio","neighbor_richness"]]
X_train = sm.add_constant(X_train, has_constant='add')

model = sm.GLM(y_train, X_train, family=sm.families.NegativeBinomial()).fit()

# approximate neighbours for test cells
test["neighbor_richness"] = train["species_richness"].mean()

X_test = test[["obs_density","migration_ratio","neighbor_richness"]]
X_test = sm.add_constant(X_test, has_constant='add')
X_test = X_test[X_train.columns]

pred = model.predict(X_test)

# ------------------------------------------------------------
# 5. Accuracy metrics
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(test["species_richness"], pred))
mae  = mean_absolute_error(test["species_richness"], pred)
r2   = r2_score(test["species_richness"], pred)

print("\nSpatial Prediction Accuracy")
print("RMSE:", round(rmse,3))
print("MAE :", round(mae,3))
print("R2  :", round(r2,3))

# ------------------------------------------------------------
# 6. Save final predictions
# ------------------------------------------------------------
gdf["residual"] = gdf["species_richness"] - gdf["predicted_richness"]
gdf.to_file("india_final_spatial_model.gpkg", driver="GPKG")

print("Saved: india_final_spatial_model.gpkg")

### RandomForest Regressor

In [ ]:
import geopandas as gpd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --------------------------------------------------
# Load data
# --------------------------------------------------
gdf = gpd.read_file("india_final_spatial_model.gpkg")

# spatial neighbors (already computed earlier but safe)
from libpysal.weights import Queen
w = Queen.from_dataframe(gdf)
w.transform = "r"

gdf["neighbor_richness"] = w.sparse @ gdf["species_richness"].values

# --------------------------------------------------
# Features & target
# --------------------------------------------------
features = ["obs_density", "migration_ratio", "neighbor_richness"]
X = gdf[features]
y = gdf["species_richness"]

# --------------------------------------------------
# Train / Test split
# --------------------------------------------------
np.random.seed(42)
mask = np.random.rand(len(gdf)) < 0.8

X_train, X_test = X[mask], X[~mask]
y_train, y_test = y[mask], y[~mask]

print("Train:", len(X_train), "Test:", len(X_test))

# --------------------------------------------------
# Train model
# --------------------------------------------------
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

# --------------------------------------------------
# Predictions
# --------------------------------------------------
pred = rf.predict(X_test)

# --------------------------------------------------
# Accuracy
# --------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("\nRandom Forest Performance")
print("RMSE:", round(rmse,3))
print("MAE :", round(mae,3))
print("R2  :", round(r2,3))

# --------------------------------------------------
# Save predictions
# --------------------------------------------------
gdf["rf_prediction"] = rf.predict(X)
gdf.to_file("india_rf_model.gpkg")

print("Saved: india_rf_model.gpkg")

### XGBoost

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# -----------------------------
# Features
# -----------------------------
features = ["obs_density", "migration_ratio", "neighbor_richness"]

X_train = train[features]
y_train = train["species_richness"]

X_test = test[features]
y_test = test["species_richness"]

# -----------------------------
# Model
# -----------------------------
gbr = GradientBoostingRegressor(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

gbr.fit(X_train, y_train)

# -----------------------------
# Prediction
# -----------------------------
pred = gbr.predict(X_test)

# -----------------------------
# Evaluation
# -----------------------------
rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("Gradient Boosting Performance")
print("RMSE:", round(rmse,3))
print("MAE :", round(mae,3))
print("R2  :", round(r2,3))